# ISOM5240 — Fine-tune Emotion Classifier
**Dataset:** `google-research-datasets/go_emotions` (28 labels → remapped to 6 music moods)
**Models:** `distilbert-base-uncased` (primary) vs `albert-base-v2` (comparison)
**Task:** Text Classification — map user Instagram text to music mood
**Label mapping:** majority vote after remapping all multi-labels to 6 music moods


In [2]:
# ── 1. Install dependencies ────────────────────────────────────────────
!pip install transformers datasets evaluate accelerate scikit-learn huggingface_hub sentencepiece -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [13]:
# ── 2. Imports ────────────────────────────────────────────────────────
import numpy as np
import time
from collections import Counter
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate
from sklearn.metrics import classification_report
from huggingface_hub import notebook_login

HF_USERNAME = 'MelodyWEN7'  # ← replace
LABEL_NAMES = ['happy', 'sad', 'romantic', 'intense', 'surprised', 'neutral']
LABEL2ID    = {l: i for i, l in enumerate(LABEL_NAMES)}
ID2LABEL    = {i: l for i, l in enumerate(LABEL_NAMES)}

# Priority order for tie-breaking
PRIORITY = ['happy', 'romantic', 'sad', 'surprised', 'intense', 'neutral']


In [5]:
# ── 3. Login to HuggingFace Hub ───────────────────────────────────────
notebook_login()


In [6]:
# ── 4. Load go_emotions dataset ───────────────────────────────────────
raw = load_dataset('google-research-datasets/go_emotions', 'simplified')
print(raw)
print('\nSample:', raw['train'][0])
print('\nOriginal label names:', raw['train'].features['labels'].feature.names)


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

Sample: {'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}

Original label names: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [14]:
# ── 5. Define label remapping (28 go_emotions → 6 music moods) ────────
original_labels = raw['train'].features['labels'].feature.names

REMAP = {
    # HAPPY
    'joy': 'happy', 'amusement': 'happy', 'excitement': 'happy',
    'pride': 'happy', 'relief': 'happy', 'gratitude': 'happy',
    'approval': 'happy', 'admiration': 'happy',
    # SAD
    'sadness': 'sad', 'grief': 'sad', 'disappointment': 'sad',
    'remorse': 'sad', 'embarrassment': 'sad',
    # ROMANTIC
    'love': 'romantic', 'caring': 'romantic',
    'desire': 'romantic', 'optimism': 'romantic',
    # INTENSE
    'anger': 'intense', 'annoyance': 'intense', 'disgust': 'intense',
    'disapproval': 'intense', 'fear': 'intense', 'nervousness': 'intense',
    # SURPRISED
    'surprise': 'surprised', 'realization': 'surprised',
    'confusion': 'surprised', 'curiosity': 'surprised',
    # NEUTRAL
    'neutral': 'neutral',
}

def remap_labels(example):
    """
    Multi-label majority vote remapping:
    1. Map each original label → music mood
    2. Count music mood frequencies
    3. Pick most frequent; break ties by PRIORITY order
    """
    mapped = [REMAP.get(original_labels[i], 'neutral') for i in example['labels']]
    if not mapped:
        final = 'neutral'
    else:
        counts = Counter(mapped)
        max_count = max(counts.values())
        candidates = [m for m in counts if counts[m] == max_count]
        # Break ties by priority
        final = next((p for p in PRIORITY if p in candidates), candidates[0])
    example['label'] = LABEL2ID[final]
    return example

remapped = raw.map(remap_labels)
print('Remapping complete.')
print('Sample after remap:', remapped['train'][0])

# Check class distribution
from collections import Counter
train_dist = Counter([ID2LABEL[x] for x in remapped['train']['label']])
print('\nClass distribution (train):')
for mood, count in sorted(train_dist.items(), key=lambda x: -x[1]):
    print(f'  {mood:12s}: {count:5d} ({count/len(remapped["train"])*100:.1f}%)')


Remapping complete.
Sample after remap: {'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej', 'label': 5}

Class distribution (train):
  happy       : 13181 (30.4%)
  neutral     : 12823 (29.5%)
  intense     :  5912 (13.6%)
  surprised   :  4418 (10.2%)
  romantic    :  4183 (9.6%)
  sad         :  2893 (6.7%)


In [15]:
# ── 6. Fine-tune helper function ──────────────────────────────────────
accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

def finetune_model(model_name, hub_name, num_epochs=3):
    print(f'\n=== Fine-tuning {model_name} ===')
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True, max_length=128)

    tokenized = remapped.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(LABEL_NAMES),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    training_args = TrainingArguments(
        output_dir                  = f'./{hub_name}',
        num_train_epochs            = num_epochs,
        per_device_train_batch_size = 32,
        per_device_eval_batch_size  = 64,
        warmup_steps                = 100,
        weight_decay                = 0.01,
        learning_rate               = 2e-5,
        evaluation_strategy         = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'accuracy',
        push_to_hub                 = True,
        hub_model_id                = f'{HF_USERNAME}/{hub_name}',
        logging_steps               = 100,
        report_to                   = 'none',
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = tokenized['train'],
        eval_dataset    = tokenized['validation'],
        tokenizer       = tokenizer,
        data_collator   = data_collator,
        compute_metrics = compute_metrics,
    )

    start = time.time()
    trainer.train()
    runtime = time.time() - start
    print(f'Training done in {runtime/60:.1f} minutes')

    # Evaluate on test set
    test_results = trainer.evaluate(tokenized['test'])
    print(f'Test accuracy: {test_results["eval_accuracy"]*100:.2f}%')

    # Detailed classification report
    preds_output = trainer.predict(tokenized['test'])
    y_pred = np.argmax(preds_output.predictions, axis=-1)
    y_true = preds_output.label_ids
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))

    trainer.push_to_hub()
    print(f'Model pushed to: https://huggingface.co/{HF_USERNAME}/{hub_name}')

    return {
        'model_name': model_name,
        'hub_name': hub_name,
        'test_accuracy': test_results['eval_accuracy'],
        'runtime_min': runtime / 60,
    }


In [16]:
# ── 7. Fine-tune Model A: distilbert-base-uncased ─────────────────────
result_distilbert = finetune_model(
    model_name = 'distilbert-base-uncased',
    hub_name   = 'distilbert-go-emotions-music',
)



=== Fine-tuning distilbert-base-uncased ===


Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [10]:
# ── 8. Fine-tune Model B: albert-base-v2 (comparison) ────────────────
result_albert = finetune_model(
    model_name = 'albert-base-v2',
    hub_name   = 'albert-go-emotions-music',
)



=== Fine-tuning albert-base-v2 ===


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.dense.bias       | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.decoder.bias     | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [17]:
# ── 9. Experiment 1: Model comparison table ───────────────────────────
from transformers import pipeline as hf_pipeline
import time

TEST_SAMPLES = [
    ('best day ever with my girls omg',          'happy'),
    ('missing summer so much it hurts',           'sad'),
    ('this view took my breath away',             'surprised'),
    ('so in love with this person',               'romantic'),
    ('i cannot believe they did that to me',      'intense'),
    ('just another day nothing special',          'neutral'),
    ('finally got the promotion so excited',      'happy'),
    ('crying watching old photos of us',          'sad'),
    ('never expected to find this place',         'surprised'),
    ('you make everything feel so warm',          'romantic'),
]

def evaluate_model(hub_name, samples):
    clf = hf_pipeline('text-classification',
                      model=f'{HF_USERNAME}/{hub_name}')
    correct = 0
    total_time = 0
    for text, true_label in samples:
        start = time.time()
        pred  = clf(text)[0]['label'].lower()
        total_time += time.time() - start
        correct += (pred == true_label)
    return {
        'accuracy':    correct / len(samples),
        'avg_time_ms': (total_time / len(samples)) * 1000,
    }

eval_distilbert = evaluate_model('distilbert-go-emotions-music', TEST_SAMPLES)
eval_albert     = evaluate_model('albert-go-emotions-music',     TEST_SAMPLES)

print('\n=== EXPERIMENT 1: MODEL SELECTION ===')
print(f'{"Model":<35} {"Accuracy":>10} {"Avg Time (ms)":>15} {"Size":>10}')
print('-' * 75)
print(f'{"distilbert-base-uncased (fine-tuned)":<35} {eval_distilbert["accuracy"]*100:>9.1f}% {eval_distilbert["avg_time_ms"]:>14.1f}ms {"268MB":>10}')
print(f'{"albert-base-v2 (fine-tuned)":<35} {eval_albert["accuracy"]*100:>9.1f}% {eval_albert["avg_time_ms"]:>14.1f}ms {"48MB":>10}')
print('\n→ Selected model for deployment: distilbert (best accuracy/size trade-off)')


OSError: MelodyWEN7/distilbert-go-emotions-music is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`